# Same-prototype control search: single-structure test

This notebook tests whether a Materials Project structure can be used to find chemically different structures with the same underlying prototype. The initial seed is **Pr2MgNi2**, a member of a structural family that includes several related compounds.

The search uses Materials Project metadata to narrow the candidates, then uses `StructureMatcher.fit_anonymous()` so element identities can differ.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from mp_api.client import MPRester
from pymatgen.analysis.structure_matcher import StructureMatcher

def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "data" / "electrides").exists():
            return path
    raise FileNotFoundError("Could not locate the project root.")

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")
MP_API_KEY = os.getenv("MP_API_KEY", "").strip()

if not MP_API_KEY:
    raise ValueError("MP_API_KEY was not found in the project .env file.")

print("Project root:", PROJECT_ROOT)
print("API key loaded:", bool(MP_API_KEY))

/opt/anaconda3/envs/electride-ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/andrewweiland/Andrews_Lab_Research/electride-ml-predictor
API key loaded: True


In [2]:
# Start with one known structural family
seed_formula = "Pr2MgNi2"

fields = [
    "material_id",
    "formula_pretty",
    "structure",
    "symmetry",
    "energy_above_hull",
    "formation_energy_per_atom",
    "volume",
    "nsites",
]

with MPRester(api_key=MP_API_KEY, mute_progress_bars=True) as mpr:
    seed_docs = mpr.materials.summary.search(
        formula=seed_formula,
        fields=fields,
    )

if not seed_docs:
    raise ValueError(f"No Materials Project entries found for {seed_formula}.")

# Use the lowest-energy entry as the initial seed structure.
seed_doc = min(
    seed_docs,
    key=lambda doc: doc.energy_above_hull if doc.energy_above_hull is not None else float("inf"),
)

seed_structure = seed_doc.structure
seed_symmetry = seed_doc.symmetry

print("Seed Materials Project ID:", seed_doc.material_id)
print("Seed formula:", seed_doc.formula_pretty)
print("Seed space group:", seed_symmetry.symbol if seed_symmetry else "not reported")
print("Seed number of sites:", seed_doc.nsites)
print("Seed entries returned:", len(seed_docs))

Seed Materials Project ID: mp-2926
Seed formula: Pr2MgNi2
Seed space group: P4/mbm
Seed number of sites: 10
Seed entries returned: 1


In [3]:
# Narrow the database search using coarse structural metadata.
# These filters are only a pre-filter; they do not prove prototype identity.

candidate_fields = [
    "material_id",
    "formula_pretty",
    "structure",
    "symmetry",
    "energy_above_hull",
    "formation_energy_per_atom",
    "volume",
    "nsites",
]

with MPRester(api_key=MP_API_KEY, mute_progress_bars=True) as mpr:
    candidate_docs = mpr.materials.summary.search(
        spacegroup_number=seed_symmetry.number,
        num_sites=seed_doc.nsites,
        fields=candidate_fields,
    )

print("Coarse candidates returned:", len(candidate_docs))

candidate_preview = pd.DataFrame([
    {
        "material_id": str(doc.material_id),
        "formula": doc.formula_pretty,
        "space_group": doc.symmetry.symbol if doc.symmetry else None,
        "nsites": doc.nsites,
        "energy_above_hull": doc.energy_above_hull,
    }
    for doc in candidate_docs
])
display(candidate_preview.head(20))

Coarse candidates returned: 322


,material_id,formula,space_group,nsites,energy_above_hull
0,mp-16496,Al(CuRe)2,P4/mbm,10,0.538467
1,mp-1079053,B2Mo3,P4/mbm,10,0.023003
2,mp-11272,Be2Nb3,P4/mbm,10,0.000000
3,mp-20723,Ca2PbAu2,P4/mbm,10,0.000000
4,mp-1078274,Ca2SnGe2,P4/mbm,10,0.009513
5,mp-11288,Ca3Hg2,P4/mbm,10,0.000000
6,mp-3382,CaSiO3,P4/mbm,10,0.243730
7,mp-20908,Ce(BC)2,P4/mbm,10,0.000000
8,mp-11965,Ce2CdPd2,P4/mbm,10,0.000000
9,mp-11964,Ce2CdPt2,P4/mbm,10,0.000000


In [4]:
# Compare the seed against each candidate while allowing the elements to differ.
anonymous_matcher = StructureMatcher(
    ltol=0.2,
    stol=0.3,
    angle_tol=5,
    primitive_cell=True,
    scale=True,
    attempt_supercell=False,
    allow_subset=False,
)

seed_reduced_formula = seed_structure.composition.reduced_formula
match_rows = []

for number, doc in enumerate(candidate_docs, start=1):
    candidate_structure = doc.structure
    candidate_reduced_formula = candidate_structure.composition.reduced_formula

    # Exclude the same composition; this search is for different-element controls.
    if candidate_reduced_formula == seed_reduced_formula:
        continue

    is_match = anonymous_matcher.fit_anonymous(seed_structure, candidate_structure)

    match_rows.append({
        "seed_material_id": str(seed_doc.material_id),
        "seed_formula": seed_reduced_formula,
        "candidate_material_id": str(doc.material_id),
        "candidate_formula": candidate_reduced_formula,
        "candidate_space_group": doc.symmetry.symbol if doc.symmetry else None,
        "candidate_nsites": doc.nsites,
        "energy_above_hull": doc.energy_above_hull,
        "anonymous_structure_match": is_match,
    })

    if number == 1 or number % 25 == 0 or number == len(candidate_docs):
        print(f"Compared {number}/{len(candidate_docs)} coarse candidates")

matches_df = pd.DataFrame(match_rows)
prototype_matches_df = (
    matches_df[matches_df["anonymous_structure_match"]]
    .sort_values(["candidate_formula", "candidate_material_id"])
    .reset_index(drop=True)
)

print("Anonymous prototype matches with different formulas:", len(prototype_matches_df))
display(prototype_matches_df)

Compared 1/322 coarse candidates
Compared 25/322 coarse candidates
Compared 50/322 coarse candidates
Compared 75/322 coarse candidates
Compared 100/322 coarse candidates
Compared 125/322 coarse candidates
Compared 150/322 coarse candidates
Compared 175/322 coarse candidates
Compared 200/322 coarse candidates
Compared 225/322 coarse candidates
Compared 250/322 coarse candidates
Compared 275/322 coarse candidates
Compared 300/322 coarse candidates
Compared 322/322 coarse candidates
Anonymous prototype matches with different formulas: 252


,seed_material_id,seed_formula,candidate_material_id,candidate_formula,candidate_space_group,candidate_nsites,energy_above_hull,anonymous_structure_match
0,mp-2926,Pr2MgNi2,mp-16496,Al(CuRe)2,P4/mbm,10,0.538467,True
1,mp-2926,Pr2MgNi2,mp-20723,Ca2PbAu2,P4/mbm,10,0.000000,True
2,mp-2926,Pr2MgNi2,mp-1078274,Ca2SnGe2,P4/mbm,10,0.009513,True
3,mp-2926,Pr2MgNi2,mp-11965,Ce2CdPd2,P4/mbm,10,0.000000,True
4,mp-2926,Pr2MgNi2,mp-11964,Ce2CdPt2,P4/mbm,10,0.000000,True
...,...,...,...,...,...,...,...,...
247,mp-2926,Pr2MgNi2,mp-1078202,Yb2SnGe2,P4/mbm,10,0.000000,True
248,mp-2926,Pr2MgNi2,mp-1079489,Zr2InAu2,P4/mbm,10,0.044373,True
249,mp-2926,Pr2MgNi2,mp-22150,Zr2InCo2,P4/mbm,10,0.000000,True
250,mp-2926,Pr2MgNi2,mp-19906,Zr2InNi2,P4/mbm,10,0.000000,True


## Interpretation

Rows in `prototype_matches_df` are **candidate structural controls**, not confirmed non-electrides. They share an anonymous structural match with the seed while having a different composition.

Before adding them to the calculation set, we should remove any known electrides, check for duplicate structures, preserve the Materials Project IDs, and record `parent_seed_material_id = seed_doc.material_id`.